# Dots and Boxes — AI vs AI (estilo da versão JSX)

Este notebook mantém a mesma ideia visual e comportamental da versão React:

- layout em **cards**
- tabuleiro central com **cores por jogador**
- painel lateral com **agentes, explicação e log**
- controles **Step**, **Autoplay** e **Reset**
- **AI 1** com heurística greedy
- **AI 2** com heurística oportunista com aleatoriedade

A implementação foi dividida em duas partes:

1. **Lógica do jogo** em `dots_and_boxes_agents.py`
2. **Interface notebook-style** com `ipywidgets + HTML/SVG`

> Para rodar o autoplay e a interface interativa, use um ambiente Jupyter com `ipywidgets` habilitado.

In [11]:
import time
from html import escape
from tornado.ioloop import PeriodicCallback

import ipywidgets as widgets
from IPython.display import display

from dots_and_boxes_agents import (
    DOTS_COLS,
    DOTS_ROWS,
    PLAYERS,
    create_initial_state,
    step_game,
)

In [19]:
CELL_SIZE = 88
MARGIN = 30
AUTO_MS = 420

def edge_key(orientation, r, c):
    return f"{orientation}-{r}-{c}"

def board_svg(state):
    board_width = MARGIN * 2 + (DOTS_COLS - 1) * CELL_SIZE
    board_height = MARGIN * 2 + (DOTS_ROWS - 1) * CELL_SIZE

    parts = [
        f'<svg viewBox="0 0 {board_width} {board_height}" style="width:100%; max-width:620px; display:block; margin:0 auto;">'
    ]

    # Boxes
    for r in range(DOTS_ROWS - 1):
        for c in range(DOTS_COLS - 1):
            key = f"{r}-{c}"
            owner = state["boxes"].get(key)
            fill = PLAYERS[owner]["box_hex"] if owner else "transparent"
            x = MARGIN + c * CELL_SIZE + 10
            y = MARGIN + r * CELL_SIZE + 10
            parts.append(
                f'<rect x="{x}" y="{y}" width="{CELL_SIZE-20}" height="{CELL_SIZE-20}" '
                f'rx="16" fill="{fill}"></rect>'
            )

    # Horizontal edges
    for r in range(DOTS_ROWS):
        for c in range(DOTS_COLS - 1):
            key = edge_key("h", r, c)
            owner = state["edges"].get(key)
            x1 = MARGIN + c * CELL_SIZE
            y1 = MARGIN + r * CELL_SIZE
            x2 = MARGIN + (c + 1) * CELL_SIZE
            y2 = y1
            stroke = PLAYERS[owner]["edge"] if owner else "#d1d5db"
            width = 10 if owner else 6
            parts.append(
                f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" '
                f'stroke="{stroke}" stroke-width="{width}" stroke-linecap="round"></line>'
            )

    # Vertical edges
    for r in range(DOTS_ROWS - 1):
        for c in range(DOTS_COLS):
            key = edge_key("v", r, c)
            owner = state["edges"].get(key)
            x1 = MARGIN + c * CELL_SIZE
            y1 = MARGIN + r * CELL_SIZE
            x2 = x1
            y2 = MARGIN + (r + 1) * CELL_SIZE
            stroke = PLAYERS[owner]["edge"] if owner else "#d1d5db"
            width = 10 if owner else 6
            parts.append(
                f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" '
                f'stroke="{stroke}" stroke-width="{width}" stroke-linecap="round"></line>'
            )

    # Dots
    for r in range(DOTS_ROWS):
        for c in range(DOTS_COLS):
            cx = MARGIN + c * CELL_SIZE
            cy = MARGIN + r * CELL_SIZE
            parts.append(f'<circle cx="{cx}" cy="{cy}" r="7" fill="#111827"></circle>')

    parts.append("</svg>")
    return "".join(parts)

def player_card(player_id, score, is_current):
    p = PLAYERS[player_id]
    ring = "box-shadow: 0 0 0 2px #cbd5e1 inset;" if is_current else ""
    bg = "#ffffff" if is_current else "#f8fafc"
    badge = (
        f'<span style="display:inline-block; margin-top:12px; padding:5px 10px; border-radius:999px; '
        f'font-size:12px; font-weight:600; background:{p["badge_bg"]}; color:{p["badge_text"]}; '
        f'border:1px solid {p["badge_border"]};">Turno atual</span>'
        if is_current else
        '<span style="display:inline-block; margin-top:12px; padding:5px 10px; border-radius:999px; '
        'font-size:12px; font-weight:600; background:#ffffff; color:#475569; border:1px solid #cbd5e1;">Aguardando</span>'
    )
    return f'''
    <div style="border:1px solid #e2e8f0; border-radius:18px; padding:16px; background:{bg}; {ring}">
        <div style="display:flex; justify-content:space-between; align-items:center; gap:12px;">
            <div style="display:flex; align-items:center; gap:12px;">
                <div style="width:16px; height:16px; border-radius:999px; background:{p["edge"]};"></div>
                <div>
                    <div style="font-size:13px; color:#64748b;">Agente</div>
                    <div style="font-size:20px; font-weight:700; color:{p["text"]};">{escape(p["name"])}</div>
                </div>
            </div>
            <div style="font-size:34px; font-weight:800; color:#0f172a;">{score}</div>
        </div>
        {badge}
    </div>
    '''

def build_html(state):
    total_edges = DOTS_ROWS * (DOTS_COLS - 1) + (DOTS_ROWS - 1) * DOTS_COLS
    played_edges = len(state["edges"])

    if state["finished"]:
        if state["winner"] == 0:
            status = "Empate"
        else:
            status = f'{PLAYERS[state["winner"]]["name"]} venceu'
    else:
        status = f'{PLAYERS[state["currentPlayer"]]["name"]} está pensando'

    log_items = "".join(
        f'<div style="border:1px solid #e2e8f0; border-radius:14px; background:#fff; padding:10px 12px; '
        f'box-shadow:0 1px 2px rgba(15,23,42,0.05); margin-bottom:8px;">{escape(item)}</div>'
        for item in state["log"][:14]
    )

    return f'''
    <div style="font-family: Inter, ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, Arial, sans-serif;
                color:#0f172a;
                background:linear-gradient(to bottom, #e2e8f0 0%, #f8fafc 45%, #ffffff 100%);
                padding:24px; border-radius:28px;">
        <div style="max-width:1400px; margin:0 auto; display:grid; grid-template-columns: minmax(0, 1.25fr) 420px; gap:24px;">
            <div style="background:#fff; border:1px solid #e2e8f0; border-radius:28px; box-shadow:0 20px 40px rgba(148,163,184,0.18); overflow:hidden;">
                <div style="padding:28px 28px 10px 28px;">
                    <div style="display:flex; justify-content:space-between; gap:20px; align-items:flex-start; flex-wrap:wrap;">
                        <div>
                            <div style="font-size:34px; font-weight:800; letter-spacing:-0.03em;">Dots and Boxes — AI vs AI</div>
                            <div style="margin-top:10px; max-width:850px; font-size:14px; line-height:1.6; color:#475569;">
                                Demo minimalista para disciplina de agentes inteligentes: dois agentes observam o mesmo ambiente,
                                escolhem ações e competem por recompensa ao fechar caixas.
                            </div>
                        </div>
                    </div>
                </div>

                <div style="padding:0 28px 28px 28px;">
                    <div style="display:grid; grid-template-columns:repeat(3, minmax(0, 1fr)); gap:16px; margin-bottom:24px;">
                        <div style="border:1px solid #e2e8f0; border-radius:18px; background:#fff; padding:16px; box-shadow:0 1px 2px rgba(15,23,42,0.05);">
                            <div style="font-size:13px; color:#64748b;">Status</div>
                            <div style="margin-top:4px; font-size:22px; font-weight:700;">{escape(status)}</div>
                        </div>
                        <div style="border:1px solid #e2e8f0; border-radius:18px; background:#fff; padding:16px; box-shadow:0 1px 2px rgba(15,23,42,0.05);">
                            <div style="font-size:13px; color:#64748b;">Contagem de jogadas</div>
                            <div style="margin-top:4px; font-size:22px; font-weight:700;">{state["turnCount"]}</div>
                        </div>
                        <div style="border:1px solid #e2e8f0; border-radius:18px; background:#fff; padding:16px; box-shadow:0 1px 2px rgba(15,23,42,0.05);">
                            <div style="font-size:13px; color:#64748b;">Arestas desenhadas</div>
                            <div style="margin-top:4px; font-size:22px; font-weight:700;">{played_edges} / {total_edges}</div>
                        </div>
                    </div>

                    <div style="border:1px solid #e2e8f0; border-radius:28px; background:#fff; padding:16px; box-shadow: inset 0 2px 10px rgba(148,163,184,0.10); overflow:auto;">
                        {board_svg(state)}
                    </div>
                </div>
            </div>

            <div style="display:flex; flex-direction:column; gap:24px;">
                <div style="background:#fff; border:1px solid #e2e8f0; border-radius:28px; box-shadow:0 10px 25px rgba(148,163,184,0.16);">
                    <div style="padding:22px 22px 10px 22px; font-size:22px; font-weight:800;">Agentes</div>
                    <div style="padding:0 22px 22px 22px; display:flex; flex-direction:column; gap:14px;">
                        {player_card(1, state["scores"][1], (not state["finished"] and state["currentPlayer"] == 1))}
                        {player_card(2, state["scores"][2], (not state["finished"] and state["currentPlayer"] == 2))}
                    </div>
                </div>

                <div style="background:#fff; border:1px solid #e2e8f0; border-radius:28px; box-shadow:0 10px 25px rgba(148,163,184,0.16); display: none;">
                    <div style="padding:22px 22px 10px 22px; font-size:22px; font-weight:800;">Por que isso se encaixa em Agentes Inteligentes</div>
                    <div style="padding:0 22px 22px 22px; font-size:14px; line-height:1.7; color:#334155;">
                        <p>Sim — esse projeto é adequado para a disciplina. O tabuleiro é o <b>ambiente</b>, cada IA é um <b>agente</b>,
                        cada turno é uma <b>seleção de ação</b> e a pontuação funciona como <b>recompensa</b>.</p>
                        <p>É mais simples do que um mundo com navegação, mas mantém o mesmo ciclo central:
                        <b>perceber estado → escolher ação → afetar ambiente → receber resultado</b>.</p>
                        <p>Para apresentação rápida, isso é excelente porque o ambiente é pequeno, totalmente observável, determinístico e fácil de explicar.</p>
                    </div>
                </div>

                <div style="background:#fff; border:1px solid #e2e8f0; border-radius:28px; box-shadow:0 10px 25px rgba(148,163,184,0.16);">
                    <div style="padding:22px 22px 10px 22px; font-size:22px; font-weight:800;">Políticas de decisão</div>
                    <div style="padding:0 22px 22px 22px; font-size:14px; line-height:1.7; color:#334155;">
                        <p>Os agentes usam políticas diferentes:</p>
                        <ul style="padding-left:20px; margin:8px 0;">
                            <li><b>AI 1</b>: heurística greedy — fecha caixas imediatamente, evita jogadas arriscadas e escolhe a aresta mais segura.</li>
                            <li><b>AI 2</b>: heurística oportunista com aleatoriedade — fecha caixas, prioriza jogadas seguras com maior pressão e desempata com aleatoriedade.</li>
                        </ul>
                        <p>Isso evita partidas totalmente rígidas sem exigir algoritmos mais pesados, como minimax.</p>
                    </div>
                </div>

                <div style="background:#fff; border:1px solid #e2e8f0; border-radius:28px; box-shadow:0 10px 25px rgba(148,163,184,0.16);">
                    <div style="padding:22px 22px 10px 22px; font-size:22px; font-weight:800;">Ações recentes</div>
                    <div style="padding:0 22px 22px 22px;">
                        <div style="max-height:280px; overflow:auto; background:#f8fafc; border-radius:18px; padding:12px;">
                            {log_items}
                        </div>
                    </div>
                </div>
            </div>
        </div>
    </div>
    '''

class NotebookDotsAndBoxesDemo:
    def __init__(self):
        self.state = create_initial_state()
        self.running = False
        self.autoplay_callback = None

        self.view = widgets.HTML()
        self.step_btn = widgets.Button(
            description="Step",
            button_style="primary",
            icon="step-forward",
            layout=widgets.Layout(width="120px", height="40px")
        )
        self.auto_btn = widgets.ToggleButton(
            description="Autoplay",
            icon="play",
            layout=widgets.Layout(width="140px", height="40px", display='none')
        )
        self.reset_btn = widgets.Button(
            description="Reset",
            icon="refresh",
            layout=widgets.Layout(width="120px", height="40px")
        )

        self.controls = widgets.HBox(
            [self.step_btn, self.auto_btn, self.reset_btn],
            layout=widgets.Layout(gap="10px", margin="0 0 14px 0")
        )

        self.step_btn.on_click(self.on_step)
        self.reset_btn.on_click(self.on_reset)
        self.auto_btn.observe(self.on_toggle_autoplay, names="value")

        self.render()

    def render(self):
        self.view.value = build_html(self.state)
        finished = self.state["finished"]
        self.step_btn.disabled = finished
        self.auto_btn.disabled = finished
        if finished and self.auto_btn.value:
            self.auto_btn.value = False

    def on_step(self, _):
        if self.state["finished"]:
            return
        self.state = step_game(self.state)
        self.render()

    def on_reset(self, _):
        self.stop_autoplay()
        self.state = create_initial_state()
        self.render()

    def on_toggle_autoplay(self, change):
        if change["new"]:
            self.start_autoplay()
        else:
            self.stop_autoplay()

    def _autoplay_tick(self):
        if not self.running or self.state["finished"]:
            self.stop_autoplay()
            self.render()
            return

        self.state = step_game(self.state)
        self.render()

        if self.state["finished"]:
            self.stop_autoplay()
            self.render()

    def start_autoplay(self):
        if self.running or self.state["finished"]:
            return

        self.running = True
        self.auto_btn.icon = "pause"
        self.auto_btn.description = "Pause"

        self.autoplay_callback = PeriodicCallback(
            self._autoplay_tick,
            AUTO_MS,
        )
        self.autoplay_callback.start()

    def stop_autoplay(self):
        self.running = False

        if self.autoplay_callback is not None:
            self.autoplay_callback.stop()
            self.autoplay_callback = None

        self.auto_btn.icon = "play"
        self.auto_btn.description = "Autoplay"

    def display(self):
        display(self.controls)
        display(self.view)

In [20]:
demo = NotebookDotsAndBoxesDemo()
demo.display()

HTML(value='\n    <div style="font-family: Inter, ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, A…